In [1]:
import yaml 
from pathlib import Path
import os
#path = Path("C:/Users/HP/Desktop/UNOS data/codes/simulation-waitline-unos").resolve()  # Get the absolute path
#path = Path("/Users/felipesimon/UMN/Research/KidneyFailure/waitlist-unos-simulation")
path = Path("C:/Users/Admin/Documents/Felipe/waitlist-unos-simulation")

os.chdir(path)
import prediction as pred 
from data import *
from simulation import * 
import copy

In [2]:
df_base = load_data()   
#yaml_file_path = "C:/Users/HP/Desktop/UNOS data/codes/simulation-waitline-unos/mapping_data.yaml"
yaml_file_path = "C:/Users/Admin/Documents/Felipe/waitlist-unos-simulation/mapping_data.yaml"
with open(yaml_file_path, "r") as file:
    loaded_data = yaml.safe_load(file)
ethcat_mapping = loaded_data["ethcat_mapping"]
blood_type_compatibility = loaded_data["blood_type_compatibility"]

In [ ]:
real_gender_df = df_base.GENDER.value_counts(normalize=True).reset_index()
real_gender_df.columns = ['gender', 'sample_proportion']
real_gender_df

In [ ]:
real_ethcat_df = df_base.ETHCAT.value_counts(normalize=True).reset_index()
real_ethcat_df.ETHCAT = real_ethcat_df.ETHCAT.map(ethcat_mapping)
real_ethcat_df.columns = ['ethcat', 'sample_proportion']
real_ethcat_df

In [ ]:
decision_tree = pred.SurvivalPredictionModel(model_path='models/decision_tree_1212.pkl')
cox_prop = pred.SurvivalPredictionModel(model_path='models/coxprop_0912.pkl')
random_forest = pred.SurvivalPredictionModel(model_path='models/forest_0912.pkl')

replicates = 30
events_per_replication =create_list_of_events(n_events = 10000, replications=replicates)

#filter events:
replicates_to_consider = [1]
events_per_replication = {k: v for k,v in events_per_replication.items() if k in replicates_to_consider}


events_per_replication_p1 = copy.deepcopy(events_per_replication)
events_per_replication_p2 = copy.deepcopy(events_per_replication)
events_per_replication_p3 = copy.deepcopy(events_per_replication)


# Survival Decision Trees

## Policy 1

In [ ]:
average_waiting_times, number_of_matches_list, results_p1_no, results_p1_ll, results_p1_fl, all_recipients = run_simulation(events_per_replication_p1, decision_tree, policy='p1', verbose=False)
df_p1_dt = all_recipients_to_dataframe(all_recipients, 'Survival_Decision_Tree', 'p1')

## Policy 2

In [ ]:
average_waiting_times, number_of_matches_list, results_p2_no, results_p2_ll, results_p2_fl, all_recipients_p2_dt = run_simulation(events_per_replication_p2, decision_tree, policy='p2', verbose=False)

df_p2_dt = all_recipients_to_dataframe(all_recipients_p2_dt, 'Survival_Decision_Tree', 'p2')

## Policy 3

In [ ]:
average_waiting_times, number_of_matches_list, results_p3_no, results_p3_ll, results_p3_fl, all_recipients_p3_dt = run_simulation(events_per_replication_p3, decision_tree, policy='p3', verbose=False)

df_p3_dt = all_recipients_to_dataframe(all_recipients_p3_dt, 'Survival_Decision_Tree', 'p3')

# Survival Random Forest

In [9]:
events_per_replication_p1 = copy.deepcopy(events_per_replication)
events_per_replication_p2 = copy.deepcopy(events_per_replication)
events_per_replication_p3 = copy.deepcopy(events_per_replication)

## Policy 1

In [ ]:
average_waiting_times, number_of_matches_list, results_p1_no, results_p1_ll, results_p1_fl, all_recipients_p1_sf = run_simulation(events_per_replication_p1, random_forest, policy='p1', verbose=False)

df_p1_sf = all_recipients_to_dataframe(all_recipients_p1_sf, 'Random_Survival_Forest', 'p1')

## Policy 3

In [ ]:
average_waiting_times, number_of_matches_list, results_p3_no, results_p3_ll, results_p3_fl, all_recipients_p3_sf = run_simulation(events_per_replication_p3, random_forest, policy='p3', verbose=False)

df_p3_sf = all_recipients_to_dataframe(all_recipients_p3_sf, 'Random_Survival_Forest', 'p3')

# Cox Proportional Hazard

In [ ]:
events_per_replication_p1 = copy.deepcopy(events_per_replication)
events_per_replication_p2 = copy.deepcopy(events_per_replication)
events_per_replication_p3 = copy.deepcopy(events_per_replication)

## Policy 1

In [ ]:
average_waiting_times, number_of_matches_list, results_p1_no, results_p1_ll, results_p1_fl, all_recipients_p1_fl = run_simulation(events_per_replication_p1, cox_prop, 
                                                                                                            policy='p1', verbose=False)

df_p1_fl = all_recipients_to_dataframe(all_recipients_p1_fl, 'Cox_Proportional_Hazard', 'p1')

## Policy 3

In [ ]:
average_waiting_times, number_of_matches_list, results_p3_no, results_p3_ll, results_p3_fl, all_recipients_p3_fl = run_simulation(events_per_replication_p3, cox_prop, 
                                                                                                            policy='p3', verbose=False)

df_p3_fl = all_recipients_to_dataframe(all_recipients_p3_fl, 'Cox_Proportional_Hazard', 'p3')

# Join results

In [ ]:
df_final = pd.concat([df_p1_dt, df_p2_dt, df_p3_dt, df_p1_sf, df_p3_sf, df_p1_fl, df_p3_fl])

df_final